# COMP4153 Natural Language Processing
# Lab 2 — Word Vectors, Semantics and Embeddings

**Version:** Student  
**Duration:** approximately 2 hours  
**Environment:** Google Colab or a local Jupyter Notebook  
**Mode:** self-guided — read the explanations and run the cells in order.

Lab 1 converted text into simple numerical features. Lab 2 goes further: instead of representing words only by IDs or sparse counts, we investigate how **vector geometry can encode semantic information**.

The first part uses the **Movie Reviews** corpus as a fully guided walkthrough. In Part B you apply the same ideas to **Twitter Samples**.

## Learning objectives

By the end of the lab, you should be able to:

- explain the difference between word IDs, one-hot vectors, count-based vectors, and dense embeddings;
- compute dot products and cosine similarity;
- construct a word-context co-occurrence matrix;
- convert co-occurrence counts to **PPMI** values;
- use **SVD** to obtain lower-dimensional count-based word vectors;
- inspect nearest neighbours and visualise word vectors;
- train a small **Skip-gram Word2Vec** model;
- explain why static embeddings have difficulty with polysemy;
- recognise that preprocessing, corpus size, domain, and projection choices affect the embedding space.

## Suggested timing

| Part | Approx. time |
|---|---:|
| 0. Setup and recap | 10 min |
| Part A — Guided Movie Reviews walkthrough | 60 min |
| Part B — Twitter embedding tasks | 50 min |

## How to work through this notebook

For each representation, keep track of **what one row means**, **how many dimensions it has**, and **what information it preserves**. This is more important than memorising library calls.

A useful comparison throughout the lab is:

| Representation | Typical shape for one word | What the dimensions mean |
|---|---:|---|
| integer ID | scalar | only an index |
| one-hot | vocabulary size | one position identifies the word |
| co-occurrence row | context-vocabulary size | counts/associations with context words |
| SVD embedding | chosen dense dimension | compressed latent features |
| Word2Vec embedding | chosen dense dimension | features learned from a local prediction objective |

When you see a new NumPy array or matrix, inspect its `.shape` and ask what each axis represents.


# 0. Setup

This notebook follows the same environment pattern as Lab 1.

- **Google Colab:** Google Drive is mounted and the notebook looks for files in `My Drive/COMP4153/Lab2`.
- **Local Jupyter:** the notebook uses the current working directory. Keep the notebook and all Lab 2 CSV files in the same folder and open Jupyter from that folder.

The required data files are:

- `movie_reviews.csv`
- `twitter_samples.csv`
No dataset is downloaded inside the notebook.

## 0.1 Import the libraries

Lab 2 uses the libraries from Lab 1 plus:

- `scikit-learn` for SVD, PCA, and normalisation;
- `gensim` for Word2Vec.

The first code cell imports the standard libraries. The following cell checks whether `gensim` is installed and installs it only if necessary.

### Library roles

`NumPy` handles vector and matrix arithmetic; `pandas` stores tabular data; `Matplotlib` produces plots; `NLTK` supplies tokenisers; `scikit-learn` provides SVD, PCA, stop words, and vector normalisation; and `gensim` trains Word2Vec.

The lab intentionally uses standard library implementations for SVD/PCA/Word2Vec so that the focus stays on the NLP concepts rather than low-level optimisation code.


In [ ]:
# Import the tools used throughout the lab.
# NumPy: vector/matrix operations; pandas: tabular data; Matplotlib: plots.
# NLTK: tokenisation; scikit-learn: dimensionality reduction/normalisation.
from pathlib import Path
from collections import Counter
import os
import random
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.tokenize import TreebankWordTokenizer, TweetTokenizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize


In [ ]:
# Try to import gensim. If the package is missing, install it into
# the current Python environment and then import Word2Vec.
try:
    import gensim
except ImportError:
    print("gensim is not installed. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim"])
    import gensim

from gensim.models import Word2Vec

print("gensim version:", gensim.__version__)


## 0.2 Detect the environment and locate the Lab 2 folder

If your Google Drive folder has a different name, change `COLAB_LAB_DIR` below.

`Path` is used rather than manually concatenating folder strings because it works cleanly across operating systems. After `os.chdir(LAB_DIR)`, a filename such as `"movie_reviews.csv"` is interpreted relative to that lab folder.


In [ ]:
# Detect Colab versus local Jupyter, choose the lab folder,
# then make that folder the current working directory.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    COLAB_LAB_DIR = Path("/content/drive/MyDrive/COMP4153/Lab2")
    LAB_DIR = COLAB_LAB_DIR
    print("Environment: Google Colab")
else:
    LAB_DIR = Path.cwd()
    print("Environment: local Jupyter Notebook")

if not LAB_DIR.exists():
    raise FileNotFoundError(f"Lab folder not found: {LAB_DIR}")

os.chdir(LAB_DIR)
print("Working directory:", Path.cwd())

## 0.3 Check the data files

A relative filename such as `movie_reviews.csv` is searched for in the current working directory. The following check stops early with a clear message if a required file is missing.

This is a **fail-fast check**: if the datasets are not in the expected location, the notebook stops immediately with a clear error instead of failing much later during preprocessing.


In [ ]:
# Check both required CSV files before any analysis begins.
# This produces a clearer error than waiting for pd.read_csv(...) to fail later.
required_files = [
    "movie_reviews.csv",
    "twitter_samples.csv",
]

for filename in required_files:
    print(f"{filename:24s}", Path(filename).exists())

missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError("Missing required file(s): " + ", ".join(missing))

In [ ]:
# Fix Python and NumPy random seeds so that stochastic steps in this
# notebook are easier to reproduce across repeated runs.
random.seed(42)
np.random.seed(42)

---
# Part A — Guided walkthrough using Movie Reviews

**There are no independent tasks in Part A.** Run the cells in order and focus on the connection between the lecture concepts and their implementation.

The two main routes explored are:

**count-based:** corpus → co-occurrence → PPMI → SVD → dense vectors  
**prediction-based:** corpus → local prediction task → Word2Vec → dense vectors

The two routes eventually produce the same kind of object—a dense vector for each word—but they arrive there differently. The count-based route first constructs an explicit word-context matrix. Word2Vec skips that explicit matrix and learns vectors directly from a prediction objective.

Throughout Part A, the sentiment labels are intentionally ignored: we are learning **word representations from text context**, not training a sentiment classifier.


## 1. From symbolic words to vector geometry

A vocabulary can map words to integer IDs, but the IDs themselves do not encode meaning. If `cat = 0` and `dog = 1`, the numerical distance between `0` and `1` is not a semantic statement.

A one-hot vector is numerical, but all different words are equally orthogonal. Dense embeddings aim for a more useful geometry: related words should tend to occupy nearby regions of the vector space.

### Why integer IDs are not embeddings

A mapping such as `cat → 0`, `dog → 1`, `bank → 2` is useful for lookup, but arithmetic on the IDs is meaningless. For example, `bank - cat = 2` does **not** mean that *bank* is twice as far from *cat* as *dog* is.

A one-hot vector removes that false numeric ordering, but it still gives every pair of different words the same dot product (zero). Semantic structure therefore has to come from the geometry of a richer vector space.


In [ ]:
# Create a toy vocabulary and map each word to an integer ID.
# np.eye(V) creates the V x V identity matrix; each row is one one-hot vector.
toy_vocab = ["cat", "dog", "bank", "river", "loan"]
toy_ids = {word: i for i, word in enumerate(toy_vocab)}

one_hot = np.eye(len(toy_vocab), dtype=int)
one_hot_df = pd.DataFrame(one_hot, index=toy_vocab, columns=toy_vocab)

print("Word IDs:", toy_ids)
display(one_hot_df)

The dot product between any two *different* one-hot vectors is zero. For example, `cat` and `dog` receive no more similarity than `cat` and `bank`.

For a vocabulary of size \(V\), a one-hot vector is \(V\)-dimensional and contains exactly one `1`. The vectors are therefore maximally sparse. The representation is good for identity, but not for graded similarity.


In [ ]:
# Select the one-hot rows for three words and compare them with a dot product.
# Different one-hot rows have no overlapping 1s, so their dot product is 0.
cat_onehot = one_hot[toy_ids["cat"]]
dog_onehot = one_hot[toy_ids["dog"]]
bank_onehot = one_hot[toy_ids["bank"]]

print("cat · dog =", np.dot(cat_onehot, dog_onehot))
print("cat · bank =", np.dot(cat_onehot, bank_onehot))

## 2. Dot product and cosine similarity

For two vectors, the **dot product** measures alignment but is affected by both direction and length. **Cosine similarity** divides by the vector lengths and therefore focuses on the angle between them:

$$\cos(\mathbf{x},\mathbf{y})= \frac{\mathbf{x}^{\top}\mathbf{y}} {\|\mathbf{x}\|\|\mathbf{y}\|}$$

We reproduce the lecture example using:

- `cat = [2, 1]`
- `dog = [2, 2]`
- `car = [0, 3]`

### Interpreting cosine similarity

Cosine similarity lies between \(-1\) and \(1\) for ordinary real-valued vectors:

- values near \(1\): similar direction;
- values near \(0\): roughly orthogonal directions;
- values near \(-1\): opposite directions.

For many word-embedding spaces, we mainly interpret **larger cosine values as greater similarity**. Cosine ignores overall scale, so two vectors can have very different lengths yet still have cosine similarity close to 1 if they point in the same direction.


In [ ]:
# Three small 2-D vectors make cosine similarity easy to inspect by hand.
# The custom function divides the dot product by the product of vector lengths.
cat = np.array([2.0, 1.0])
dog = np.array([2.0, 2.0])
car = np.array([0.0, 3.0])

def cosine_similarity(x, y):
    denominator = np.linalg.norm(x) * np.linalg.norm(y)
    if denominator == 0:
        return 0.0
    return float(np.dot(x, y) / denominator)

print("cat · dog =", np.dot(cat, dog))
print("cos(cat, dog) =", round(cosine_similarity(cat, dog), 3))
print("cos(cat, car) =", round(cosine_similarity(cat, car), 3))

`cat` and `dog` have the smaller angle and therefore the larger cosine similarity.

The next plot visualises the three vectors as arrows from the origin. A 2D picture is useful for intuition, but real embeddings typically have tens or hundreds of dimensions.

The important point is not the particular toy numbers. It is that cosine compares **direction**, whereas the raw dot product is also affected by vector magnitude.


In [ ]:
# Draw each 2-D vector as an arrow from the origin.
# Similar directions correspond to smaller angles and therefore larger cosine values.
vectors_2d = {"cat": cat, "dog": dog, "car": car}

fig, ax = plt.subplots(figsize=(6, 5))
for word, vec in vectors_2d.items():
    ax.arrow(0, 0, vec[0], vec[1], head_width=0.10, length_includes_head=True)
    ax.text(vec[0] + 0.08, vec[1] + 0.08, word)

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlim(-0.5, 3.2)
ax.set_ylim(-0.5, 3.5)
ax.set_xlabel("dimension 1")
ax.set_ylabel("dimension 2")
ax.set_title("Cosine similarity as vector direction")
plt.show()

## 3. The distributional hypothesis and co-occurrence

The distributional hypothesis says that words appearing in similar contexts tend to have similar meanings.

We first use the same tiny corpus as the lecture:

1. `cats chase mice`
2. `dogs chase cats`
3. `dogs eat food`

With a context window of 1, the context of a centre word consists of its immediate left and right neighbours.

### Centre words and context words

With window size 1, the sentence

```text
cats chase mice
```

produces the local relations `cats ↔ chase` and `chase ↔ mice`. With a larger window, more distant words are also counted.

The window is therefore a modelling choice: a small window tends to emphasise local syntactic/functional relationships, while a larger window can capture broader topical relationships.


In [ ]:
# Represent the tiny corpus as a list of sentences, where each sentence
# is already tokenised into a list of strings.
tiny_sentences = [
    ["cats", "chase", "mice"],
    ["dogs", "chase", "cats"],
    ["dogs", "eat", "food"],
]

tiny_vocab = ["cats", "dogs", "chase", "mice", "eat", "food"]
tiny_index = {word: i for i, word in enumerate(tiny_vocab)}
print(tiny_index)

The helper function below constructs a word-context matrix.

- rows are **target/centre words**;
- columns are **context words**;
- each cell counts how often the pair occurs within the selected window.

If the vocabulary has \(V\) words, the matrix has shape \(V $\times$ V\). Row \(i\) corresponds to a centre word and column \(j\) to a context word. In this implementation the same vocabulary is used for rows and columns, but that is a design choice rather than a mathematical requirement.

The nested loops implement a sliding context window. `left` and `right` prevent the window from running outside the sentence boundaries.


In [ ]:
# Build a target-word x context-word matrix.
# The same vocabulary is used for rows and columns, so the result has shape [V, V].
def build_cooccurrence(sentences, word_to_index, window_size=1):
    size = len(word_to_index)
    matrix = np.zeros((size, size), dtype=np.float32)

    for sentence in sentences:
        ids = [word_to_index[w] for w in sentence if w in word_to_index]

        for i, centre_id in enumerate(ids):
            left = max(0, i - window_size)
            right = min(len(ids), i + window_size + 1)

            for j in range(left, right):
                if j != i:
                    context_id = ids[j]
                    matrix[centre_id, context_id] += 1.0

    return matrix

In [ ]:
# Apply the co-occurrence helper with a one-word context window,
# then wrap the matrix in a DataFrame so row/column words are visible.
tiny_counts = build_cooccurrence(
    tiny_sentences,
    tiny_index,
    window_size=1
)

tiny_counts_df = pd.DataFrame(
    tiny_counts.astype(int),
    index=tiny_vocab,
    columns=tiny_vocab
)

display(tiny_counts_df)

For example, `chase` occurs next to `cats` twice, once in each of the first two sentences, so the corresponding count is 2.

Because the context window is symmetric here, the matrix will often be symmetric as well: if `cats` appears in the context of `chase`, then `chase` also appears in the context of `cats`. This would not necessarily hold for a directional context definition.


In [ ]:
# Visualise the count matrix as a heatmap.
# Brighter/higher-valued cells correspond to more frequent target-context pairs.
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(tiny_counts)
ax.set_xticks(range(len(tiny_vocab)), tiny_vocab, rotation=45, ha="right")
ax.set_yticks(range(len(tiny_vocab)), tiny_vocab)
ax.set_xlabel("context word")
ax.set_ylabel("target word")
ax.set_title("Tiny-corpus co-occurrence matrix")
fig.colorbar(image, ax=ax, label="count")
plt.show()

## 4. From raw counts to PPMI

Raw counts can be dominated by generally frequent words. **Pointwise Mutual Information (PMI)** asks whether a word-context pair occurs more often than expected from the individual frequencies of the word and context.

$$
PMI(w,c)=\log_2\frac{P(w,c)}{P(w)P(c)}
$$

**Positive PMI (PPMI)** clips negative values to zero:

$$
PPMI(w,c)=\max(PMI(w,c),0).
$$

The following function applies this calculation to a count matrix.

In [ ]:
# Convert raw co-occurrence counts to Positive PMI (PPMI).
# Only cells with positive counts are evaluated; negative PMI values are clipped to zero.
def ppmi_matrix(count_matrix):
    counts = count_matrix.astype(np.float64)
    total = counts.sum()

    row_totals = counts.sum(axis=1, keepdims=True)
    col_totals = counts.sum(axis=0, keepdims=True)
    expected = row_totals @ col_totals

    ppmi = np.zeros_like(counts)
    nonzero = counts > 0

    ppmi[nonzero] = np.log2(
        (counts[nonzero] * total) / expected[nonzero]
    )

    ppmi = np.maximum(ppmi, 0.0)
    return ppmi

In [ ]:
# Apply PPMI to the toy matrix and display rounded values with the
# original vocabulary labels so each association remains interpretable.
tiny_ppmi = ppmi_matrix(tiny_counts)

tiny_ppmi_df = pd.DataFrame(
    np.round(tiny_ppmi, 2),
    index=tiny_vocab,
    columns=tiny_vocab
)

display(tiny_ppmi_df)

PPMI values are **association strengths**, not raw frequencies. A high value means that the target and context occur together more often than expected by chance.

A zero PPMI value can mean either that the pair never occurred or that it did not occur more often than expected. Therefore PPMI should be interpreted as a **positive association measure**, not as a probability.


## 5. Build count-based embeddings from Movie Reviews

We now repeat the count-based route on a real corpus.

The sentiment labels are **not used** here. Embedding learning is based only on the text and its local contexts.

For this example we:

- lowercase;
- keep alphabetic tokens;
- remove a standard English stop-word list to make content-word relationships easier to inspect;
- keep the 1,000 most frequent remaining words;
- use a context window of 2.

These choices are not universal rules. Changing preprocessing changes the resulting vector space.

The preprocessing here is deliberately simple and transparent. Removing stop words can make content-word relations easier to inspect in a small corpus, but it also discards information. Real systems should choose preprocessing according to the downstream objective rather than applying stop-word removal automatically.

The resulting `movie_sentences` is a list of token lists, which is exactly the input structure required by both the co-occurrence helper and `gensim.models.Word2Vec`.


In [ ]:
# Load the real Movie Reviews corpus.
# The sentiment labels are present in the CSV but are not used to learn embeddings.
movie_df = pd.read_csv("movie_reviews.csv")

print("Shape:", movie_df.shape)
display(movie_df.head(2))

In [ ]:
# Define one preprocessing function for Movie Reviews.
# It lowercases, tokenises, keeps alphabetic content words, and removes stop words.
treebank_tokenizer = TreebankWordTokenizer()

def tokenize_for_embeddings(text):
    tokens = treebank_tokenizer.tokenize(str(text).lower())
    return [
        token for token in tokens
        if token.isalpha()
        and len(token) > 1
        and token not in ENGLISH_STOP_WORDS
    ]

In [ ]:
# Tokenise every document, then flatten all retained tokens into one Counter.
# The nested generator yields tokens sentence by sentence without making a second huge list.
movie_sentences = [
    tokenize_for_embeddings(text)
    for text in movie_df["text"]
]

movie_counter = Counter(
    token
    for sentence in movie_sentences
    for token in sentence
)

print("Documents:", len(movie_sentences))
print("Total retained tokens:", sum(movie_counter.values()))
print("Distinct retained tokens:", len(movie_counter))
print("Most common:", movie_counter.most_common(15))

We restrict the co-occurrence matrix to the top 1,000 words. A full matrix over every type would be much larger and much sparser.

For a 1,000-word vocabulary, each count-based word representation initially has 1,000 dimensions. The matrix therefore has shape `[1000, 1000]`: 1,000 target words by 1,000 context features.


In [ ]:
# Keep the 1,000 most frequent retained tokens for the count-based vocabulary.
# enumerate(...) creates a stable word -> row/column index mapping.
movie_vocab_words = [
    word for word, count in movie_counter.most_common(1000)
]
movie_word_to_index = {
    word: i for i, word in enumerate(movie_vocab_words)
}

print("Count-based vocabulary size:", len(movie_vocab_words))
print("First 15 words:", movie_vocab_words[:15])

In [ ]:
# Count how often vocabulary words occur near one another using window size 2.
# Rows are centre words; columns are context words.
movie_counts = build_cooccurrence(
    movie_sentences,
    movie_word_to_index,
    window_size=2
)

print("Co-occurrence matrix shape:", movie_counts.shape)
print("Non-zero cells:", np.count_nonzero(movie_counts))
print("Total counted word-context pairs:", int(movie_counts.sum()))

A 1,000 × 1,000 matrix has one million cells. Many are still zero, so it is a high-dimensional sparse representation.

We now apply PPMI and then reduce the matrix to 50 dimensions with **Truncated SVD**.

### Why SVD is useful here

SVD approximates the high-dimensional PPMI matrix with a lower-dimensional representation. We keep 50 components, so every 1,000-dimensional PPMI row becomes a 50-dimensional dense vector.

The 50 dimensions are **latent features**: unlike the original columns, they do not correspond directly to named context words. Their value comes from preserving important structure in a compact space.


In [ ]:
# Reweight the 1,000 x 1,000 raw count matrix using PPMI.
# The matrix shape stays the same; only the cell values change.
movie_ppmi = ppmi_matrix(movie_counts)

print("PPMI matrix shape:", movie_ppmi.shape)
print("Non-zero PPMI cells:", np.count_nonzero(movie_ppmi))

In [ ]:
# Reduce each 1,000-D PPMI row to 50 latent dimensions with Truncated SVD.
# L2-normalise each 50-D row so dot products can be read as cosine similarities.
svd = TruncatedSVD(n_components=50, random_state=42)
movie_count_vectors = svd.fit_transform(movie_ppmi)

# Unit normalisation makes cosine similarity a simple dot product.
movie_count_vectors = normalize(movie_count_vectors)

print("Dense embedding shape:", movie_count_vectors.shape)

The transformation is now:

**1,000-dimensional sparse PPMI row → 50-dimensional dense vector**

Each row of `movie_count_vectors` is the dense vector for the word at the same row position in `movie_vocab_words`.

After SVD, the vectors are L2-normalised so that each row has length approximately 1. This is convenient because, for unit-length vectors,

$$
\mathbf{x}^{\top}\mathbf{y} = \cos(\mathbf{x},\mathbf{y}).
$$

That is why the nearest-neighbour function can use a matrix-vector dot product instead of calling a separate cosine function for every word.


In [ ]:
# Retrieve the row belonging to 'good'.
# Its 50 values are the dense count-based embedding learned from the corpus.
word = "good"
word_id = movie_word_to_index[word]

print("Word:", word)
print("Vector shape:", movie_count_vectors[word_id].shape)
print("First 10 values:", np.round(movie_count_vectors[word_id][:10], 3))

## 6. Nearest neighbours in the count-based embedding space

Nearest neighbours are words with the largest cosine similarity to a query vector.

Because the SVD vectors were normalised, their matrix dot product with a normalised query vector gives cosine similarities.

The similarity computation is vectorised: `movie_count_vectors @ query_vector` compares the query with **all vocabulary vectors at once**. The result is a one-dimensional similarity array with one score per vocabulary word.

The query word itself has similarity approximately 1, so the helper deliberately skips its own row before returning neighbours.


In [ ]:
# Find nearest neighbours by comparing one normalised query vector with
# every normalised vocabulary vector in a single matrix-vector multiplication.
def count_neighbours(word, topn=8):
    if word not in movie_word_to_index:
        raise KeyError(f"{word!r} is not in the count-based vocabulary")

    query_id = movie_word_to_index[word]
    query_vector = movie_count_vectors[query_id]
    similarities = movie_count_vectors @ query_vector
    order = np.argsort(-similarities)

    neighbours = []
    for idx in order:
        if idx != query_id:
            neighbours.append(
                (movie_vocab_words[idx], float(similarities[idx]))
            )
        if len(neighbours) == topn:
            break

    return neighbours

In [ ]:
# Inspect several query words. Each returned tuple is
# (neighbour_word, cosine_similarity_score).
for query in ["good", "actor", "funny", "horror", "love"]:
    print(f"\nNearest neighbours of {query!r}:")
    print(count_neighbours(query, topn=6))

Notice that nearest neighbours are often **related** words rather than exact synonyms. Distributional similarity captures patterns of contextual use. Antonyms can also become close because they may occur in similar contexts.

Nearest neighbours should be interpreted as evidence about the **corpus and representation**, not as a dictionary definition of synonymy. A word may be close to an antonym, a common collocate, or a word from the same topic because those words can appear in similar contexts.


## 7. Visualise selected count-based embeddings in 2D

PCA projects the 50-dimensional vectors into two dimensions for plotting. This is useful for intuition, but the projection necessarily discards information and can distort distances.

### PCA is a visualisation, not the embedding itself

The model still has 50-dimensional vectors. PCA creates a temporary two-dimensional projection only so that humans can plot selected words on a page.

Distances in the 2D plot can be distorted, especially when many dimensions are compressed into only two. Use the plot to explore patterns, but use the original 50-dimensional vectors for quantitative similarity.


In [ ]:
# Keep only requested words that exist in the vocabulary, stack their
# 50-D vectors, then fit a 2-D PCA projection for visualisation.
plot_words = [
    "good", "great", "excellent", "bad", "terrible",
    "actor", "actress", "role", "funny", "comedy",
    "horror", "thriller", "love", "relationship"
]
plot_words = [w for w in plot_words if w in movie_word_to_index]

plot_matrix = np.stack([
    movie_count_vectors[movie_word_to_index[w]]
    for w in plot_words
])

coords = PCA(n_components=2).fit_transform(plot_matrix)

In [ ]:
# Plot the 2-D PCA coordinates and annotate each point with its word.
# These coordinates are a projection, not the original embedding vectors.
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[:, 0], coords[:, 1])

for word, (x, y) in zip(plot_words, coords):
    ax.text(x + 0.01, y + 0.01, word)

ax.set_xlabel("PCA dimension 1")
ax.set_ylabel("PCA dimension 2")
ax.set_title("Selected Movie Reviews count-based embeddings")
plt.show()

## 8. Prediction-based embeddings with Word2Vec

Count-based embeddings explicitly build co-occurrence statistics. **Word2Vec** instead learns vectors through a prediction task based on local word windows.

We train a small **Skip-gram** model:

- `sg=1`: Skip-gram (centre word predicts surrounding words);
- `vector_size=50`: each word receives a 50-dimensional vector;
- `window=5`: use nearby words within five positions;
- `min_count=5`: ignore very rare words;
- `epochs=5`: make five passes through the training corpus;
- `workers=1`: improves reproducibility for this example.

No sentiment labels are used.

### What Skip-gram learns

Skip-gram creates training examples from a sliding window. A centre word is used to predict nearby context words. Repeated over a corpus, words that are useful for predicting similar contexts tend to acquire similar vectors.

`Word2Vec` contains more machinery internally than shown here (including the output-side parameters used during training), but after training we normally access the learned word vectors through `model.wv`.


In [ ]:
# Train a Skip-gram Word2Vec model directly from tokenised sentences.
# sg=1 selects Skip-gram; vector_size=50 determines the learned vector width.
movie_w2v = Word2Vec(
    sentences=movie_sentences,
    vector_size=50,
    window=5,
    min_count=5,
    workers=1,
    sg=1,
    epochs=5,
    seed=42,
)

print("Word2Vec vocabulary size:", len(movie_w2v.wv))
print("Embedding dimension:", movie_w2v.wv.vector_size)

In [ ]:
# .wv gives access to the learned input-side word vectors.
# similarity(...) computes cosine similarity between two stored word vectors.
print("First 10 values for 'good':")
print(np.round(movie_w2v.wv["good"][:10], 3))

print("\ncos(good, great) =", round(movie_w2v.wv.similarity("good", "great"), 3))
print("cos(good, movie) =", round(movie_w2v.wv.similarity("good", "movie"), 3))

In [ ]:
# most_similar(...) compares the query with the Word2Vec vocabulary
# and returns the highest-cosine neighbours.
for query in ["good", "funny", "horror", "actor", "love"]:
    print(f"\nWord2Vec neighbours of {query!r}:")
    print(movie_w2v.wv.most_similar(query, topn=6))

Word2Vec and PPMI+SVD may produce different neighbours because they learn representations in different ways. Both, however, rely on the distributional idea that context provides information about meaning.

The two methods are not expected to produce identical neighbours. PPMI+SVD compresses an explicit association matrix; Word2Vec optimises a prediction objective. Different hyperparameters, preprocessing, random initialisation, and corpus size can all change the geometry.


## 9. The limitation of static embeddings: polysemy

Consider:

- `She deposited money in the bank.`
- `They sat by the river bank.`

A static Word2Vec model has only one lookup vector for the word type `bank`. The sentence in which the word occurs does not change the stored vector.

### Type-level versus token-level representations

A **word type** is the vocabulary entry `bank`. A **token** is one occurrence of `bank` in a particular sentence. Static embeddings store one vector per **type**, so every token of `bank` retrieves the same vector.

Contextual models instead construct a vector for each **token occurrence**, allowing the representation to depend on surrounding words.


In [ ]:
# Demonstrate the limitation of a static lookup table.
# Both variables retrieve exactly the same stored row for the word type 'bank'.
if "bank" in movie_w2v.wv:
    bank_financial = movie_w2v.wv["bank"]
    bank_river = movie_w2v.wv["bank"]

    print("Same stored vector:", np.allclose(bank_financial, bank_river))
    print("Cosine similarity:", cosine_similarity(bank_financial, bank_river))
else:
    print("'bank' is not in this small Word2Vec vocabulary.")

A **contextual embedding** model instead creates a representation for each token occurrence using the surrounding sentence. Therefore the two occurrences of `bank` can receive different vectors.

We do not download a large contextual language model in this two-hour lab. The purpose here is to make the static/contextual distinction concrete; contextual models will be studied in later parts of the module.

This distinction prepares for later neural language models. Contextual representations are especially useful for polysemous words because the same spelling can express different senses in different sentences.


---
# Part B — Coding tasks using Twitter Samples

You will now apply the same embedding workflow to the second standard corpus. The Twitter sentiment labels are used only for a quick dataset overview; the embedding methods themselves learn from the text without sentiment labels.

Part B contains **coding tasks only**. Reuse and adapt the code patterns from Part A whenever guidance points you to a section.

The workflow is:

**inspect Twitter data → preprocess text → co-occurrence counts → PPMI → SVD embeddings → nearest neighbours → PCA → Skip-gram Word2Vec → compare the two embedding spaces**

Part B deliberately mirrors Part A. You are not expected to invent a new algorithm; the goal is to recognise which **dataset-specific objects** must be replaced while keeping the same conceptual pipeline.

The Twitter corpus also introduces domain-specific text phenomena such as hashtags, user handles, emoticons, and repeated characters. This is why `TweetTokenizer` is used instead of simply reusing the Movie Reviews tokeniser.


In [ ]:
# Load the second corpus used for the independent coding tasks in Part B.
twitter_df = pd.read_csv("twitter_samples.csv")

print("Shape:", twitter_df.shape)
display(twitter_df.head())


In [ ]:
# TweetTokenizer is tailored to social-media text.
# Here it lowercases, reduces exaggerated repetitions, and removes @user handles.
tweet_tokenizer = TweetTokenizer(
    preserve_case=False,
    reduce_len=True,
    strip_handles=True
)


## Task B1 — Explore the Twitter dataset

1. display the label counts;
2. check missing values in `text` and `label`;
3. create a `whitespace_length` column using whitespace-separated words;
4. print the mean and median tweet length;
5. plot a histogram of tweet lengths.

**Guidance:** refer to the Lab 1 / pandas pattern using `value_counts()`, `isna().sum()`, `.str.split().str.len()`, and `.plot(kind="hist", ...)`.

**Before coding:** distinguish *dataset inspection* from *embedding training*. The label counts are useful for understanding the corpus, but the labels are not supplied to the embedding algorithms.

For the length histogram, `whitespace_length` is only a quick descriptive measure; it is not the same as the number of tokens retained after the embedding preprocessing rules.


In [ ]:
# Task B1a
# Display label counts and missing values.

# Suggested steps:
# - use value_counts() on the label column;
# - select ["text", "label"], then use isna().sum().

# YOUR CODE HERE

In [ ]:
# Task B1b
# Create whitespace_length, report mean/median,
# and plot a histogram of tweet lengths.

# Suggested steps:
# - create whitespace_length using .str.split().str.len();
# - compute mean() and median();
# - call .plot(kind="hist", ...) and label the x-axis.

# YOUR CODE HERE

## Task B2 — Prepare the Twitter corpus for embeddings

Define `tokenize_tweet_for_embeddings(text)` using `tweet_tokenizer`.

Use these preprocessing rules:

1. tokenise the text;
2. remove the `#` symbol from a hashtag but keep the hashtag word;
3. keep alphabetic tokens only;
4. keep tokens longer than one character;
5. remove tokens in `ENGLISH_STOP_WORDS`.

Then create `twitter_sentences`, build `twitter_counter`, and print:

- number of tokenised tweets;
- total number of retained tokens;
- number of distinct retained token types;
- 20 most frequent tokens.

**Guidance:** refer to **Part A, Section 5**, especially `tokenize_for_embeddings`, the construction of `movie_sentences`, and `Counter(...)`.

**Expected data flow:** one raw tweet becomes one cleaned token list. The complete corpus therefore becomes a `list[list[str]]`.

When handling hashtags, removing only the leading `#` allows `#happy` to contribute the lexical item `happy`. Because `strip_handles=True` was used when creating `TweetTokenizer`, user handles are removed by the tokeniser.


In [ ]:
# Task B2a
# Define tokenize_tweet_for_embeddings(text).
# Refer to Part A, Section 5.

# Suggested steps:
# - call tweet_tokenizer.tokenize(str(text));
# - loop over the tokens;
# - if a token begins with '#', remove only that first symbol;
# - apply the alphabetic/length/stop-word conditions before appending.

# YOUR CODE HERE

In [ ]:
# Task B2b
# Create twitter_sentences and twitter_counter,
# then print the requested corpus statistics.

# Suggested steps:
# - use a list comprehension to apply your tokeniser to every tweet;
# - build Counter(token for sentence in ... for token in sentence);
# - use Counter values/length/most_common(...) for the requested summaries.

# YOUR CODE HERE

## Task B3 — Build a Twitter co-occurrence matrix

1. select the 1,000 most frequent Twitter tokens as `twitter_vocab_words`;
2. create `twitter_word_to_index`;
3. use `build_cooccurrence(...)` with `window_size=2` to create `twitter_counts`;
4. print the matrix shape, the number of non-zero cells, and the percentage of non-zero cells.

**Guidance:** refer to **Part A, Section 5**. Reuse the existing `build_cooccurrence(...)` function; you do not need to redefine it.

After Task B3, check that `twitter_counts.shape == (1000, 1000)` when at least 1,000 vocabulary items are available. `np.count_nonzero(...)` measures matrix sparsity: a low non-zero percentage is normal because most word pairs never occur within the selected window.


In [ ]:
# Task B3a
# Create twitter_vocab_words and twitter_word_to_index.

# Suggested steps:
# - take twitter_counter.most_common(1000);
# - keep only the word from each (word, count) pair;
# - use enumerate(...) to build word -> index.

# YOUR CODE HERE

In [ ]:
# Task B3b
# Build twitter_counts with window_size=2
# and report matrix sparsity.

# Suggested steps:
# - reuse build_cooccurrence(..., window_size=2);
# - use np.count_nonzero(matrix);
# - divide non-zero cells by matrix.size and multiply by 100.

# YOUR CODE HERE

## Task B4 — Convert Twitter counts to dense PPMI + SVD embeddings

1. convert `twitter_counts` to `twitter_ppmi` using `ppmi_matrix(...)`;
2. reduce the PPMI matrix to 50 dimensions with `TruncatedSVD`;
3. L2-normalise the resulting vectors;
4. print the PPMI shape and dense-vector shape.

**Guidance:** refer to **Part A, Section 5**. Reuse `ppmi_matrix(...)`, then follow the `TruncatedSVD(n_components=50, random_state=42)` and `normalize(...)` pattern used for Movie Reviews.

The PPMI matrix and SVD vectors should preserve row alignment: row `i` must continue to represent `twitter_vocab_words[i]`. Losing this alignment would make every later similarity result meaningless.


In [ ]:
# Task B4a
# Convert twitter_counts to twitter_ppmi.

# Suggested step:
# Reuse ppmi_matrix(twitter_counts), then inspect shape and non-zero values.

# YOUR CODE HERE

In [ ]:
# Task B4b
# Apply TruncatedSVD with 50 components and L2-normalise
# the dense Twitter count vectors.

# Suggested steps:
# - create TruncatedSVD(n_components=50, random_state=42);
# - call fit_transform(twitter_ppmi);
# - pass the result through normalize(...).

# YOUR CODE HERE

## Task B5 — Find nearest neighbours in the Twitter count-based embedding space

Define `twitter_count_neighbours(word, topn=6)` by adapting the Movie Reviews helper. Then print the six nearest neighbours for `good`, `happy`, `sad`, and `love` when the query word is available.

**Guidance:** refer to **Part A, Section 6**. Adapt `count_neighbours(...)` by replacing the Movie Reviews vocabulary, index, and vector variables with the Twitter versions.

Because the vectors are normalised, the same efficient pattern from Part A applies:

```text
all vectors [V, 50] @ query [50] → similarities [V]
```

Then sort the similarity scores from largest to smallest, skip the query itself, and return the first `topn` neighbours.


In [ ]:
# Task B5a
# Define twitter_count_neighbours(word, topn=6).
# Refer to Part A, Section 6.

# Suggested steps:
# Follow count_neighbours(...) from Part A, replacing the Movie Reviews
# vocabulary, index, and vector variables with the Twitter versions.

# YOUR CODE HERE

In [ ]:
# Task B5b
# Print nearest neighbours for the requested query words.

# Suggested steps:
# Loop over the requested query words, check vocabulary membership,
# then print twitter_count_neighbours(query, topn=6).

# YOUR CODE HERE

## Task B6 — Visualise selected Twitter count-based embeddings with PCA

Project the available vectors for the following words into two dimensions and plot them:

`happy, birthday, sad, tired, good, great, love, thanks, today, tomorrow`

**Guidance:** refer to **Part A, Section 7**. Use the same sequence: filter words that are in the vocabulary → `np.stack(...)` their vectors → `PCA(n_components=2).fit_transform(...)` → `ax.scatter(...)` → label each point with `ax.text(...)`.

Only words that actually occur in the selected Twitter vocabulary can be plotted. Filtering the requested list before `np.stack(...)` prevents a vocabulary lookup error.

Again, interpret proximity in the PCA plot cautiously: it is a two-dimensional view of a 50-dimensional space.


In [ ]:
# Task B6a
# Select available plot words, stack their count-based vectors,
# and reduce them to 2D with PCA.

# Suggested steps:
# Filter the requested words by vocabulary membership, stack their 50-D
# vectors with np.stack(...), then fit PCA(n_components=2).

# YOUR CODE HERE

In [ ]:
# Task B6b
# Draw and label the PCA scatter plot.

# Suggested steps:
# Use ax.scatter(...) for the PCA coordinates, then zip(words, coords)
# and annotate each point with ax.text(...).

# YOUR CODE HERE

## Task B7 — Train a Twitter Skip-gram Word2Vec model

Train `twitter_w2v` with:

- `vector_size=50`;
- `window=5`;
- `min_count=3`;
- `workers=1`;
- `sg=1`;
- `epochs=10`;
- `seed=42`.

Then print the Word2Vec vocabulary size and embedding dimension.

**Guidance:** refer to **Part A, Section 8** and adapt the `movie_w2v = Word2Vec(...)` cell.

The Twitter Word2Vec settings differ slightly from the Movie Reviews example (`min_count=3`, `epochs=10`) because the corpora differ. `min_count` controls vocabulary coverage; `epochs` controls how many passes are made over the training examples.


In [ ]:
# Task B7
# Train twitter_w2v using the specified Skip-gram parameters
# and print vocabulary size and embedding dimension.

# Suggested step:
# Adapt the Word2Vec constructor from Part A using exactly the parameter
# values specified in Task B7, then inspect len(model.wv) and vector_size.

# YOUR CODE HERE

## Task B8 — Compute Word2Vec cosine similarities and nearest neighbours

1. calculate cosine similarity for `good`–`great`, `happy`–`sad`, and `happy`–`birthday` when both words are available;
2. print the six nearest Word2Vec neighbours of `happy`, `sad`, and `thanks` when available.

**Guidance:** refer to **Part A, Section 8**. Use `twitter_w2v.wv.similarity(w1, w2)` and `twitter_w2v.wv.most_similar(query, topn=6)`.

A high cosine similarity does not necessarily imply synonymy. For example, positive and negative sentiment words can still be distributionally related if they occur in similar sentence patterns. Always interpret similarities in relation to the corpus.


In [ ]:
# Task B8a
# Calculate the requested Word2Vec cosine similarities.

# Suggested steps:
# Store the requested word pairs, check that both words are in twitter_w2v.wv,
# and call .similarity(w1, w2).

# YOUR CODE HERE

In [ ]:
# Task B8b
# Print Word2Vec nearest neighbours for the requested queries.

# Suggested steps:
# Loop over the requested query words, check membership, and call
# twitter_w2v.wv.most_similar(query, topn=6).

# YOUR CODE HERE

## Task B9 — Compare count-based and Word2Vec nearest neighbours

For each available query in `good`, `happy`, `sad`, and `love`:

1. obtain the top five neighbours from `twitter_count_neighbours(...)`;
2. obtain the top five neighbours from `twitter_w2v.wv.most_similar(...)`;
3. print both lists and the words appearing in both neighbour sets.

**Guidance:** refer to **Part A, Sections 6 and 8**. Extract only the word from each `(word, similarity)` pair before finding the set intersection.

The overlap calculation is a simple way to see whether two independently constructed embedding spaces retrieve some of the same neighbours. A small overlap is not automatically a failure: the methods use different objectives and can capture different aspects of context.


In [ ]:
# Task B9
# Compare the top-five count-based and Word2Vec neighbours
# and print their overlap for each available query.

# Suggested steps:
# - obtain top-five neighbour tuples from both methods;
# - extract only the word from each tuple;
# - convert the word lists to sets and take their intersection.

# YOUR CODE HERE

# Wrap-up

In Lab 2 you moved through the main representation levels covered in this practical:

**word IDs → one-hot vectors → co-occurrence counts → PPMI → SVD embeddings → Skip-gram Word2Vec**

You also connected these representations to **cosine similarity**, nearest neighbours, PCA visualisation, and the static/contextual distinction.

Key points to retain:

- numerical IDs are useful indices but do not encode semantic closeness;
- cosine similarity compares vector direction rather than raw magnitude;
- count-based representations use word-context co-occurrence statistics;
- PPMI highlights associations that are stronger than expected from individual frequencies;
- SVD compresses sparse count vectors into dense lower-dimensional vectors;
- Skip-gram Word2Vec learns embeddings through a self-supervised local prediction task;
- static embeddings store one vector per word type and therefore handle polysemy poorly;
- preprocessing, corpus, domain, training settings, and 2D projection choices affect what we observe.

> **Before finishing:** save your notebook. If you reconnected to a new runtime, rerun the required cells in order before continuing your work.

## Shape summary

A useful final check is to connect the main objects:

```text
co-occurrence counts:       [V, V]
PPMI matrix:                [V, V]
SVD count-based vectors:    [V, 50]
one Word2Vec vector:        [50]
PCA coordinates for k words:[k, 2]
```

Here \(V=1000\) for the count-based vocabulary. Keeping these shapes in mind will make later neural-network tensor shapes easier to follow.
